In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

<details>
<summary>Metadata — Descriptions</summary>

<details><summary>#Patient Identifier</summary><pre>#Identifier to uniquely specify a patient.</pre></details>

<details><summary>American Joint Committee on Cancer Metastasis Stage Code</summary><pre>Code to 
                                                              represent the 
                                                              defined absence 
                                                              or presence of 
                                                              distant spread 
                                                              or metastases 
                                                              (M) to 
                                                              locations via 
                                                              vascular 
                                                              channels or 
                                                              lymphatics 
                                                              beyond the 
                                                              regional lymph 
                                                              nodes, using 
                                                              criteria 
                                                              established by 
                                                              the American 
                                                              Joint Committee 
                                                              on Cancer 
                                                              (AJCC).</pre></details>

<details><summary>American Joint Committee on Cancer Publication Version Type</summary><pre>The version 
                                                                 or edition 
                                                                 of the 
                                                                 American 
                                                                 Joint 
                                                                 Committee on 
                                                                 Cancer 
                                                                 Cancer 
                                                                 Staging 
                                                                 Handbooks, a 
                                                                 publication 
                                                                 by the group 
                                                                 formed for 
                                                                 the purpose 
                                                                 of 
                                                                 developing a 
                                                                 system of 
                                                                 clinical 
                                                                 staging for 
                                                                 cancer that 
                                                                 is 
                                                                 acceptable 
                                                                 to the 
                                                                 American 
                                                                 medical 
                                                                 profession 
                                                                 and is 
                                                                 compatible 
                                                                 with other 
                                                                 accepted 
                                                                 classifications.</pre></details>

<details><summary>American Joint Committee on Cancer Tumor Stage Code</summary><pre>Code of pathological 
                                                         T (primary tumor) to 
                                                         define the size or 
                                                         contiguous extension 
                                                         of the primary tumor 
                                                         (T), using staging 
                                                         criteria from the 
                                                         American Joint 
                                                         Committee on Cancer 
                                                         (AJCC).</pre></details>

<details><summary>Birth from Initial Pathologic Diagnosis Date</summary><pre>Time interval from a 
                                                  "persons date of birth to "
                                                  the date of initial 
                                                  pathologic diagnosis, 
                                                  represented as a calculated 
                                                  number of days.</pre></details>

<details><summary>Diagnosis Age</summary><pre>Age at which a condition or disease was first diagnosed.</pre></details>

<details><summary>Disease Free (Months)</summary><pre>Disease free (months) since initial treatment.</pre></details>

<details><summary>Disease Free Status</summary><pre>Disease free status since initial treatment.</pre></details>

<details><summary>Disease-specific Survival status</summary><pre>The time period usually begins at the 
                                      time of diagnosis or at the start of 
                                      treatment and ends at the time of 
                                      death.</pre></details>

<details><summary>Ethnicity Category</summary><pre>The text for reporting information about ethnicity.</pre></details>

<details><summary>Form completion date</summary><pre>Form completion date</pre></details>

<details><summary>Genetic Ancestry Label</summary><pre>Genetic ancestries were determined using five 
                            different methods as described in Carrot-Zhang et 
                            al (2020). These consensus calls were created 
                            based on the ancestral population that received 
                            the majority of assignments for each patient.</pre></details>

<details><summary>ICD-10 Classification</summary><pre>10th revision of the International Statistical 
                           Classification of Diseases and Related Health 
                           Problems.</pre></details>

<details><summary>In PanCan Pathway Analysis</summary><pre>Patient Part of PanCan Pathway Analysis</pre></details>

<details><summary>Informed consent verified</summary><pre>Informed consent verified</pre></details>

</details>

# Decisions
---
* Weight 100% null -> drop
* DAYS-TO-INITIAL_PATHOLOGIC_DIAGNOSIS is uniformly distributed -> drop
* INFORMED_CONSENT_VERIFIED and HISTORY_NEOADJUVANT are constant -> drop
* the four months features are all strongly correlated features -> keep only PFS-months as it's the most correlated with target.
* PFS-months is strongly correlated with days-last-follow-up -> drop days-last-follow-up (less correlated with target)
* DAYS-TO-BIRTH is strongly correlated with DIAGNOSIS_AGE -> drop Age (less correlated with target)
> not sure if we should drop the other, the correlation difference is small to the third decimal, but I think it's better to keep the one with less categories and less range for gredient issues (AGE) as it might be more informative.
* Form Completion Date is not informative and logically won't help but rather may overfit -> drop
* Drop columns related to future repond (leakage) -> DFS_STATUS, DSS_STATUS
---
* race and ethnicity have almost zero correlation with target -> I will keep maybe nulls (other) categories help
* All categorical features have low correlation with target and some of them have a lot of categories -> Don't know if i should drop them.
---
* ordinal fetures -> label encoding
    * Assumed AJCC stages Across A, B, C with same number are equivalent, mitigating overfit.
    * Assumed any null would be unknown category, and thus encoded as 0, which is less than the lowest known category, to preserve ordinality and to nor contribute to the feature weight.
    * .

## Columns to drop

In [2]:
PATH = "../../../data/processed/stad_clinical_patient.parquet"
df =pd.read_parquet(PATH)

df.drop(columns = ["WEIGHT", "DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS",
                   "INFORMED_CONSENT_VERIFIED", "HISTORY_NEOADJUVANT_TRTYN",
                   "DAYS_LAST_FOLLOWUP",
                   "OS_MONTHS", "DSS_MONTHS", "DFS_MONTHS",
                   "DAYS_TO_BIRTH", "FORM_COMPLETION_DATE",
                   "DSS_STATUS", "DFS_STATUS", "OS_STATUS",
                   ], inplace = True)

df.head(2)

,PATIENT_ID,SUBTYPE,AGE,SEX,AJCC_PATHOLOGIC_TUMOR_STAGE,AJCC_STAGING_EDITION,ETHNICITY,ICD_10,ICD_O_3_HISTOLOGY,ICD_O_3_SITE,...,PERSON_NEOPLASM_CANCER_STATUS,PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT,PRIOR_DX,RACE,RADIATION_THERAPY,IN_PANCANPATHWAYS_FREEZE,PFS_STATUS,PFS_MONTHS,GENETIC_ANCESTRY_LABEL,TMB
0,TCGA-3M-AB46,STAD_CIN,70.0,Male,STAGE IB,6TH,Hispanic Or Latino,C16.5,8140/3,C16.5,...,Tumor Free,Yes,No,White,Yes,Yes,0:CENSORED,58.026761,,5.600000
1,TCGA-3M-AB47,STAD_GS,51.0,Male,STAGE IIIB,6TH,Not Hispanic Or Latino,C16.9,8140/3,C16.9,...,With Tumor,Yes,No,White,Yes,Yes,1:PROGRESSION,12.986159,AFR,3.566667


In [3]:
numeric_features = df.select_dtypes(include=np.number).columns.tolist()

categorical_features = df.select_dtypes(include="str").columns.tolist()

ordinal_features = ['AJCC_PATHOLOGIC_TUMOR_STAGE', 'PATH_T_STAGE','PATH_N_STAGE', 'PATH_M_STAGE', 'AJCC_STAGING_EDITION', 'PERSON_NEOPLASM_CANCER_STATUS']

categorical_features = [f for f in categorical_features if f not in ordinal_features]
df.shape[1] - len(categorical_features)-len(numeric_features)-len(ordinal_features)

0

## Nulls

In [4]:
(df.isna().sum()).sort_values(ascending=False)

ETHNICITY                                     116
NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT        63
PERSON_NEOPLASM_CANCER_STATUS                  58
RACE                                           58
SUBTYPE                                        53
RADIATION_THERAPY                              41
PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT     20
AJCC_PATHOLOGIC_TUMOR_STAGE                    18
AGE                                             4
PFS_MONTHS                                      3
PATH_N_STAGE                                    1
SEX                                             0
ICD_10                                          0
AJCC_STAGING_EDITION                            0
PATIENT_ID                                      0
PATH_M_STAGE                                    0
PATH_T_STAGE                                    0
ICD_O_3_HISTOLOGY                               0
ICD_O_3_SITE                                    0
PRIOR_DX                                        0


In [5]:
df[df.NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT.isna()]

,PATIENT_ID,SUBTYPE,AGE,SEX,AJCC_PATHOLOGIC_TUMOR_STAGE,AJCC_STAGING_EDITION,ETHNICITY,ICD_10,ICD_O_3_HISTOLOGY,ICD_O_3_SITE,...,PERSON_NEOPLASM_CANCER_STATUS,PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT,PRIOR_DX,RACE,RADIATION_THERAPY,IN_PANCANPATHWAYS_FREEZE,PFS_STATUS,PFS_MONTHS,GENETIC_ANCESTRY_LABEL,TMB
3,TCGA-B7-5818,STAD_EBV,62.0,Male,STAGE IB,7TH,Not Hispanic Or Latino,C16.2,8145/3,C16.2,...,Tumor Free,Yes,No,White,No,Yes,0:CENSORED,11.703981,EUR,11.500000
21,TCGA-BR-4292,STAD_MSI,73.0,Female,NaN,6TH,Not Hispanic Or Latino,C16.2,8140/3,C16.2,...,NaN,NaN,No,White,NaN,Yes,0:CENSORED,0.000000,EUR,43.633333
22,TCGA-BR-4294,STAD_CIN,65.0,Male,NaN,6TH,Not Hispanic Or Latino,C16.2,8211/3,C16.2,...,NaN,NaN,No,White,NaN,Yes,0:CENSORED,0.000000,EUR,0.933333
23,TCGA-BR-4357,STAD_CIN,58.0,Male,NaN,6TH,Not Hispanic Or Latino,C16.2,8140/3,C16.2,...,NaN,NaN,No,White,NaN,Yes,0:CENSORED,0.000000,EUR,7.533333
24,TCGA-BR-4361,STAD_MSI,66.0,Female,STAGE IIIA,6TH,Not Hispanic Or Latino,C16.9,8140/3,C16.9,...,NaN,Yes,No,White,NaN,Yes,0:CENSORED,0.000000,EUR,76.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,TCGA-VQ-A8PZ,NaN,56.0,Female,STAGE II,6TH,NaN,C16.3,8490/3,C16.3,...,Tumor Free,Yes,No,White,Yes,No,0:CENSORED,73.412894,EUR,2.433333
398,TCGA-VQ-A91A,STAD_CIN,67.0,Male,STAGE IIIB,7TH,NaN,C16.2,8480/3,C16.2,...,Tumor Free,Yes,No,White,Yes,Yes,1:PROGRESSION,4.241049,EUR,4.000000
405,TCGA-VQ-A91U,STAD_CIN,78.0,Male,STAGE IIIA,6TH,NaN,C16.0,8144/3,C16.0,...,NaN,Yes,No,Asian,No,Yes,0:CENSORED,1.709570,EAS,4.366667
410,TCGA-VQ-A91Z,STAD_CIN,67.0,Female,STAGE IIIA,6TH,NaN,C16.3,8211/3,C16.3,...,With Tumor,Yes,No,White,No,Yes,1:PROGRESSION,43.068021,EUR,4.133333


In [6]:
df.fillna(dict(
    AGE = df.AGE.median(),
    PFS_MONTHS = df.PFS_MONTHS.median(),

    PATH_N_STAGE = "NX",
    PATH_M_STAGE = "MX",

    AJCC_PATHOLOGIC_TUMOR_STAGE = "unknown",
    NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT = "unknown",
    PERSON_NEOPLASM_CANCER_STATUS = "unknown",
    SUBTYPE = "unknown",

    RACE = "other",
    ETHNICITY = "other",

    PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT = "not assessed",
    RADIATION_THERAPY = "No",
), inplace=True)

,PATIENT_ID,SUBTYPE,AGE,SEX,AJCC_PATHOLOGIC_TUMOR_STAGE,AJCC_STAGING_EDITION,ETHNICITY,ICD_10,ICD_O_3_HISTOLOGY,ICD_O_3_SITE,...,PERSON_NEOPLASM_CANCER_STATUS,PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT,PRIOR_DX,RACE,RADIATION_THERAPY,IN_PANCANPATHWAYS_FREEZE,PFS_STATUS,PFS_MONTHS,GENETIC_ANCESTRY_LABEL,TMB
0,TCGA-3M-AB46,STAD_CIN,70.0,Male,STAGE IB,6TH,Hispanic Or Latino,C16.5,8140/3,C16.5,...,Tumor Free,Yes,No,White,Yes,Yes,0:CENSORED,58.026761,,5.600000
1,TCGA-3M-AB47,STAD_GS,51.0,Male,STAGE IIIB,6TH,Not Hispanic Or Latino,C16.9,8140/3,C16.9,...,With Tumor,Yes,No,White,Yes,Yes,1:PROGRESSION,12.986159,AFR,3.566667
2,TCGA-B7-5816,STAD_MSI,51.0,Female,STAGE IIB,7TH,Not Hispanic Or Latino,C16.2,8145/3,C16.2,...,Tumor Free,Yes,No,White,No,Yes,0:CENSORED,26.695598,EUR,40.100000
3,TCGA-B7-5818,STAD_EBV,62.0,Male,STAGE IB,7TH,Not Hispanic Or Latino,C16.2,8145/3,C16.2,...,Tumor Free,Yes,No,White,No,Yes,0:CENSORED,11.703981,EUR,11.500000
4,TCGA-B7-A5TI,STAD_MSI,52.0,Male,STAGE IIIC,7TH,Not Hispanic Or Latino,C16.1,8145/3,C16.1,...,Tumor Free,Yes,No,White,No,Yes,0:CENSORED,19.561429,EUR,18.433333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431,TCGA-VQ-AA6I,STAD_CIN,68.0,Male,STAGE IIIB,7TH,other,C16.0,8140/3,C16.0,...,With Tumor,Yes,No,White,No,Yes,1:PROGRESSION,15.616267,,2.933333
432,TCGA-VQ-AA6J,STAD_CIN,75.0,Male,STAGE IIIB,7TH,other,C16.0,8144/3,C16.0,...,Tumor Free,Yes,No,Black or African American,No,Yes,0:CENSORED,27.550383,AFR_ADMIX,4.200000
433,TCGA-VQ-AA6K,STAD_CIN,59.0,Male,STAGE IIIC,6TH,other,C16.0,8490/3,C16.0,...,With Tumor,Yes,No,White,Yes,Yes,1:PROGRESSION,10.914949,EUR,2.500000
434,TCGA-ZA-A8F6,STAD_CIN,71.0,Male,STAGE IB,7TH,Not Hispanic Or Latino,C16.3,8144/3,C16.3,...,unknown,Yes,No,White,No,Yes,0:CENSORED,17.260085,EUR,1.500000


## Ordinal features Encoding

In [7]:
df[ordinal_features]

,AJCC_PATHOLOGIC_TUMOR_STAGE,PATH_T_STAGE,PATH_N_STAGE,PATH_M_STAGE,AJCC_STAGING_EDITION,PERSON_NEOPLASM_CANCER_STATUS
0,STAGE IB,T2B,N0,MX,6TH,Tumor Free
1,STAGE IIIB,T3,N2,MX,6TH,With Tumor
2,STAGE IIB,T4A,N0,M0,7TH,Tumor Free
3,STAGE IB,T2,N0,M0,7TH,Tumor Free
4,STAGE IIIC,T4,N3,M0,7TH,Tumor Free
...,...,...,...,...,...,...
431,STAGE IIIB,T3,N3,M0,7TH,With Tumor
432,STAGE IIIB,T4A,N2,M0,7TH,Tumor Free
433,STAGE IIIC,T4A,N3A,M0,6TH,With Tumor
434,STAGE IB,T2,N0,MX,7TH,unknown


In [8]:
print("unique ordinal features: ", {col: df[col].unique() for col in ordinal_features})

unique ordinal features:  {'AJCC_PATHOLOGIC_TUMOR_STAGE': <ArrowStringArray>
[  'STAGE IB', 'STAGE IIIB',  'STAGE IIB', 'STAGE IIIC', 'STAGE IIIA',
    'unknown',   'STAGE II',  'STAGE III',   'STAGE IV',  'STAGE IIA',
   'STAGE IA',    'STAGE I']
Length: 12, dtype: str, 'PATH_T_STAGE': <ArrowStringArray>
['T2B', 'T3', 'T4A', 'T2', 'T4', 'TX', 'T2A', 'T1', 'T1B', 'T4B', 'T1A']
Length: 11, dtype: str, 'PATH_N_STAGE': <ArrowStringArray>
['N0', 'N2', 'N3', 'NX', 'N1', 'N3A', 'N3B']
Length: 7, dtype: str, 'PATH_M_STAGE': <ArrowStringArray>
['MX', 'M0', 'M1']
Length: 3, dtype: str, 'AJCC_STAGING_EDITION': <ArrowStringArray>
['6TH', '7TH', '5TH', '4TH']
Length: 4, dtype: str, 'PERSON_NEOPLASM_CANCER_STATUS': <ArrowStringArray>
['Tumor Free', 'With Tumor', 'unknown']
Length: 3, dtype: str}


In [9]:
stad_ordinal_map = {
    'AJCC_PATHOLOGIC_TUMOR_STAGE': {
        'Stage I': 1, 'Stage IA': 1, 'Stage IB': 1,
        'Stage II': 2, 'Stage IIA': 2, 'Stage IIB': 2,
        'Stage III': 3, 'Stage IIIA': 3, 'Stage IIIB': 3, 'Stage IIIC': 3,
        'Stage IV': 4, 'Stage IVA': 4, 'Stage IVB': 4, "unknown":0
    },
    'AJCC_STAGING_EDITION': {
        '6th': 1, '7th': 2, '8th': 3
    },
    'PATH_T_STAGE': {
        'T1': 1, 'T1a': 1, 'T1b': 1,
        'T2': 2, 'T2a': 2, 'T2b': 2,
        'T3': 3, 'T4': 4, 'T4a': 4, 'T4b': 4
    },
    'PATH_N_STAGE': {
        'N0': 1, 'N1': 2, 'N2': 3, 'N3': 4, 'N3a': 4, 'N3b': 4, "NX":0
    },
    'PATH_M_STAGE': {
        'M0': 1, 'M1': 2, 'MX': 0  # MX is "cannot be assessed", effectively a Null
    },
    'PERSON_NEOPLASM_CANCER_STATUS': {
        'Tumor Free': 1,
        'With Tumor': 2, 'unknown':0
    }
}

def encode_clinical_ordinal(df, mapping_dict):
    """
    Applies manual mapping to specific columns and ensures
    Nulls/Unknowns are encoded as 0.
    """
    df_encoded = df.copy()

    for col, mapping in mapping_dict.items():
        if col in df_encoded.columns:
            # .map() replaces values with dict values.
            # Non-matching values and NaNs become NaN, which we then fill with 0.
            df_encoded[col] = df_encoded[col].map(mapping).fillna(0).astype(int)

    return df_encoded

In [10]:
df_encoded = encode_clinical_ordinal(df, stad_ordinal_map)
df_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 436 entries, 0 to 435
Data columns (total 24 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   PATIENT_ID                                  436 non-null    str    
 1   SUBTYPE                                     436 non-null    str    
 2   AGE                                         436 non-null    float64
 3   SEX                                         436 non-null    str    
 4   AJCC_PATHOLOGIC_TUMOR_STAGE                 436 non-null    int64  
 5   AJCC_STAGING_EDITION                        436 non-null    int64  
 6   ETHNICITY                                   436 non-null    str    
 7   ICD_10                                      436 non-null    str    
 8   ICD_O_3_HISTOLOGY                           436 non-null    str    
 9   ICD_O_3_SITE                                436 non-null    str    
 10  NEW_TUMOR_EVENT_AFTER_INI

In [11]:
df_encoded = pd.get_dummies(df_encoded, columns=categorical_features[1:])
df_encoded.shape

(436, 65)

## Target

In [12]:
y = np.log1p(df_encoded.TMB+1)
x = df_encoded.drop(columns=["TMB", "PATIENT_ID"])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape, y_train.shape)

(348, 63) (348,)


## Feature Selection

In [13]:
# Pipeline ensures scaling + model together
pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('lasso', Lasso())
])

# Hyperparameter grid
param_grid = {
    'lasso__alpha': [0.001, 0.01, 0.1, 1, 10],
    'lasso__max_iter': [1000, 5000, 10000],
    'lasso__tol': [1e-4, 1e-3, 1e-2],
    'lasso__selection': ['cyclic', 'random']
}

# Grid search
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid.fit(x_train, y_train)

print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

Best params: {'lasso__alpha': 0.01, 'lasso__max_iter': 5000, 'lasso__selection': 'random', 'lasso__tol': 0.01}
Best score: -0.2993685135251539


In [14]:
r2_score(y_test, grid.predict(x_test))

0.7023563656512732

In [19]:
best_lasso = grid.best_estimator_.named_steps['lasso']
coefficients = best_lasso.coef_

selected_features = np.where(coefficients != 0)[0]
print("Selected feature indices:", selected_features, f"\ttotaling {len(selected_features)}")

Selected feature indices: [ 0  5  8  9 10 11 12 17 19 21 26 27 28 33 35 40 43 52 53 56 57] 	totaling 21


In [20]:
x.columns

Index(['AGE', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION',
       'PATH_M_STAGE', 'PATH_N_STAGE', 'PATH_T_STAGE',
       'PERSON_NEOPLASM_CANCER_STATUS', 'PFS_MONTHS', 'SUBTYPE_STAD_CIN',
       'SUBTYPE_STAD_EBV', 'SUBTYPE_STAD_GS', 'SUBTYPE_STAD_MSI',
       'SUBTYPE_STAD_POLE', 'SUBTYPE_unknown', 'SEX_Female', 'SEX_Male',
       'ETHNICITY_Hispanic Or Latino', 'ETHNICITY_Not Hispanic Or Latino',
       'ETHNICITY_other', 'ICD_10_C16.0', 'ICD_10_C16.1', 'ICD_10_C16.2',
       'ICD_10_C16.3', 'ICD_10_C16.5', 'ICD_10_C16.9',
       'ICD_O_3_HISTOLOGY_8140/3', 'ICD_O_3_HISTOLOGY_8144/3',
       'ICD_O_3_HISTOLOGY_8145/3', 'ICD_O_3_HISTOLOGY_8211/3',
       'ICD_O_3_HISTOLOGY_8255/3', 'ICD_O_3_HISTOLOGY_8260/3',
       'ICD_O_3_HISTOLOGY_8480/3', 'ICD_O_3_HISTOLOGY_8490/3',
       'ICD_O_3_SITE_C16.0', 'ICD_O_3_SITE_C16.1', 'ICD_O_3_SITE_C16.2',
       'ICD_O_3_SITE_C16.3', 'ICD_O_3_SITE_C16.5', 'ICD_O_3_SITE_C16.9',
       'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT_No',
       '

In [26]:
selected_features_names = x.columns[selected_features].tolist()
selected_features_names

['AGE',
 'PATH_T_STAGE',
 'SUBTYPE_STAD_CIN',
 'SUBTYPE_STAD_EBV',
 'SUBTYPE_STAD_GS',
 'SUBTYPE_STAD_MSI',
 'SUBTYPE_STAD_POLE',
 'ETHNICITY_Not Hispanic Or Latino',
 'ICD_10_C16.0',
 'ICD_10_C16.2',
 'ICD_O_3_HISTOLOGY_8144/3',
 'ICD_O_3_HISTOLOGY_8145/3',
 'ICD_O_3_HISTOLOGY_8211/3',
 'ICD_O_3_SITE_C16.0',
 'ICD_O_3_SITE_C16.2',
 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT_Yes',
 'PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT_Yes',
 'RADIATION_THERAPY_No',
 'RADIATION_THERAPY_Yes',
 'PFS_STATUS_0:CENSORED',
 'PFS_STATUS_1:PROGRESSION']

In [30]:
df_selected = df_encoded[["PATIENT_ID", "TMB"]+selected_features_names]
df_selected

,PATIENT_ID,TMB,AGE,PATH_T_STAGE,SUBTYPE_STAD_CIN,SUBTYPE_STAD_EBV,SUBTYPE_STAD_GS,SUBTYPE_STAD_MSI,SUBTYPE_STAD_POLE,ETHNICITY_Not Hispanic Or Latino,...,ICD_O_3_HISTOLOGY_8145/3,ICD_O_3_HISTOLOGY_8211/3,ICD_O_3_SITE_C16.0,ICD_O_3_SITE_C16.2,NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT_Yes,PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT_Yes,RADIATION_THERAPY_No,RADIATION_THERAPY_Yes,PFS_STATUS_0:CENSORED,PFS_STATUS_1:PROGRESSION
0,TCGA-3M-AB46,5.600000,70.0,0,True,False,False,False,False,False,...,False,False,False,False,False,True,False,True,True,False
1,TCGA-3M-AB47,3.566667,51.0,3,False,False,True,False,False,True,...,False,False,False,False,True,True,False,True,False,True
2,TCGA-B7-5816,40.100000,51.0,0,False,False,False,True,False,True,...,True,False,False,True,False,True,True,False,True,False
3,TCGA-B7-5818,11.500000,62.0,2,False,True,False,False,False,True,...,True,False,False,True,False,True,True,False,True,False
4,TCGA-B7-A5TI,18.433333,52.0,4,False,False,False,True,False,True,...,True,False,False,False,False,True,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431,TCGA-VQ-AA6I,2.933333,68.0,3,True,False,False,False,False,False,...,False,False,True,False,True,True,True,False,False,True
432,TCGA-VQ-AA6J,4.200000,75.0,0,True,False,False,False,False,False,...,False,False,True,False,False,True,True,False,True,False
433,TCGA-VQ-AA6K,2.500000,59.0,0,True,False,False,False,False,False,...,False,False,True,False,True,True,False,True,False,True
434,TCGA-ZA-A8F6,1.500000,71.0,2,True,False,False,False,False,True,...,False,False,False,False,False,True,True,False,True,False


In [32]:
df_selected.to_parquet("../../../data/processed/stad_clinical_patient_selected.parquet", engine="pyarrow")